# Buổi 3: Module, tệp và thư viện NumPy, Pandas cơ bản
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Import module, cài thư viện bằng pip
- Đọc/ghi tệp văn bản và CSV
- Tính toán trên mảng NumPy
- Tạo, xem, chọn, lọc, sắp xếp và thống kê mô tả DataFrame Pandas

**Dữ liệu dùng trong buổi:** `supermarket_sales.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
# Chạy ô này trước tiên
import numpy as np
import pandas as pd
DATA = "../data/"          # Colab: DATA = "/content/data/"
pd.set_option("display.max_columns", 30)

## Phần A. Bài tập có hướng dẫn

### A1. Module chuẩn và đọc/ghi tệp văn bản
Thư viện chuẩn có sẵn (`math`, `datetime`, `csv`, `json`...). Thư viện ngoài cài bằng `pip install ten_thu_vien`
(trong notebook: `!pip install ten_thu_vien`). Dùng `with open(...)` để tệp tự đóng sau khi dùng.

In [ ]:
from datetime import date, timedelta
hom_nay = date(2025, 10, 16)
print("Hạn thanh toán (30 ngày):", hom_nay + timedelta(days=30))

with open("ghi_chu.txt", "w", encoding="utf-8") as f:
    f.write("Báo cáo ngày 16/10\n")
    f.write("Doanh thu: 125 triệu\n")
with open("ghi_chu.txt", encoding="utf-8") as f:
    for dong in f:
        print(dong.strip())

### A2. NumPy: tính toán trên cả mảng cùng lúc
Mảng NumPy cho phép tính trên toàn bộ dữ liệu mà không cần vòng lặp (vector hóa) – nhanh và gọn.

In [ ]:
gia = np.array([250, 189, 1200, 99, 560])      # nghìn đồng
sl  = np.array([10, 25, 3, 40, 8])
doanh_thu = gia * sl
print("Doanh thu từng SP:", doanh_thu)
print("Tổng:", doanh_thu.sum(), "| TB:", doanh_thu.mean(), "| Độ lệch chuẩn:", doanh_thu.std().round(1))
print("SP có doanh thu > 3000:", np.where(doanh_thu > 3000)[0])
print("Tỷ trọng (%):", (doanh_thu / doanh_thu.sum() * 100).round(1))

### A3. Đọc dữ liệu bán hàng vào DataFrame
`DataFrame` là bảng dữ liệu (dòng × cột). Luôn làm 4 việc sau khi đọc dữ liệu:
`head()` xem vài dòng, `shape` kích thước, `info()` kiểu dữ liệu và giá trị thiếu, `describe()` thống kê mô tả.

In [ ]:
df = pd.read_csv(DATA + "supermarket_sales.csv")
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe().round(1)

**Giải thích các cột:** đơn giá, tổng tiền, giá vốn, lợi nhuận gộp tính bằng **nghìn đồng**; `tong_tien` đã gồm VAT 8%; `danh_gia` là điểm khách chấm (thang 10).

### A4. Chọn cột, chọn dòng, lọc
- Một cột: `df["cot"]` (trả về Series); nhiều cột: `df[["c1", "c2"]]`
- Theo vị trí: `df.iloc[0:5]`; theo nhãn/điều kiện: `df.loc[dieu_kien, ["c1", "c2"]]`
- Lọc nhiều điều kiện: dùng `&` (và), `|` (hoặc), mỗi điều kiện trong ngoặc tròn

In [ ]:
print(df["nhom_hang"].unique())
df[["ma_hoa_don", "nhom_hang", "tong_tien"]].head(3)

In [ ]:
loc_hcm_dien_tu = df[(df["thanh_pho"] == "TP.HCM") & (df["nhom_hang"] == "Điện tử")]
print("Số hóa đơn Điện tử tại TP.HCM:", len(loc_hcm_dien_tu))
df.loc[df["tong_tien"] > 8000, ["ngay", "thanh_pho", "nhom_hang", "tong_tien"]].head()

In [ ]:
# isin: thuộc một danh sách giá trị
df[df["thanh_toan"].isin(["Thẻ", "Ví điện tử"])].shape

### A5. Tạo cột mới, sắp xếp, đếm và thống kê nhanh

In [ ]:
df["doanh_thu_truoc_vat"] = (df["tong_tien"] / 1.08).round(1)
df["bien_ln_gop"] = (df["loi_nhuan_gop"] / df["doanh_thu_truoc_vat"]).round(3)
df["don_lon"] = df["tong_tien"] >= 5000

print(df["nhom_hang"].value_counts())
print("\nTỷ lệ đơn lớn:", df["don_lon"].mean().round(3))
df.sort_values("tong_tien", ascending=False).head(5)[["ma_hoa_don", "nhom_hang", "so_luong", "tong_tien"]]

In [ ]:
# Thống kê nhanh theo nhóm (sẽ học kỹ ở buổi 5)
df.groupby("thanh_pho")["tong_tien"].sum().sort_values(ascending=False)

### A6. Ghi kết quả ra CSV / Excel

In [ ]:
top_don = df.nlargest(20, "tong_tien")
top_don.to_csv("top20_don_hang.csv", index=False, encoding="utf-8-sig")   # utf-8-sig để Excel đọc đúng tiếng Việt
top_don.to_excel("top20_don_hang.xlsx", index=False)
print("Đã ghi tệp.")

## Phần B. Bài tập tự làm

Dùng DataFrame `df` đã đọc ở Phần A.

### Bài B1. Tổng quan dữ liệu

Cho biết: số hóa đơn, khoảng thời gian dữ liệu (ngày đầu – ngày cuối), tổng doanh thu (tổng `tong_tien`, quy ra tỷ đồng),
tổng lợi nhuận gộp, giá trị đơn trung bình và điểm đánh giá trung bình.

In [ ]:
# Viết code của bạn ở đây


### Bài B2. Lọc dữ liệu theo yêu cầu quản lý

(a) Các hóa đơn của **khách thành viên** ở **Hà Nội** thanh toán bằng **ví điện tử** – có bao nhiêu hóa đơn, tổng tiền bao nhiêu?
(b) 5 hóa đơn có điểm đánh giá thấp nhất (hiển thị mã hóa đơn, chi nhánh, nhóm hàng, điểm đánh giá).
(c) Số hóa đơn mua từ 8 sản phẩm trở lên.

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Doanh thu theo nhóm hàng

Tính tổng doanh thu, số hóa đơn và tỷ trọng doanh thu (%) của từng nhóm hàng, sắp xếp giảm dần.
Nhóm nào đóng góp nhiều nhất?

<details><summary>💡 Gợi ý</summary>

`df.groupby('nhom_hang')['tong_tien'].sum()` cho tổng; chia cho tổng chung để ra tỷ trọng.

</details>

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Cột tính toán mới

Tạo cột `muc_danh_gia` nhận giá trị "Tốt" nếu `danh_gia` ≥ 8, "Trung bình" nếu từ 6 đến dưới 8, "Kém" nếu < 6.
Đếm số hóa đơn theo từng mức và tính tỷ lệ phần trăm.

<details><summary>💡 Gợi ý</summary>

Dùng `np.where(dieu_kien, gia_tri_dung, gia_tri_sai)` lồng nhau, hoặc `pd.cut`.

</details>

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Xuất báo cáo chi nhánh

Với mỗi thành phố, lọc dữ liệu và ghi ra tệp CSV riêng `bao_cao_<thành phố>.csv` (dùng vòng lặp).
In số dòng của mỗi tệp đã ghi.

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Khung giờ vàng

Từ cột `gio` (dạng "HH:MM"), tạo cột `khung_gio` (lấy giờ dạng số nguyên). Khung giờ nào có tổng doanh thu cao nhất?
Chia ngày thành Sáng (8–11), Trưa (12–14), Chiều (15–17), Tối (18–21) và so sánh doanh thu trung bình mỗi hóa đơn.

In [ ]:
# Viết code của bạn ở đây


### Bài C2. NumPy: mô phỏng giá cổ phiếu

Mô phỏng giá một cổ phiếu trong 250 phiên: giá ban đầu 50.000 đ, lợi suất mỗi phiên ~ phân phối chuẩn
trung bình 0,05%, độ lệch chuẩn 2% (`np.random.default_rng(0).normal`). Dùng `np.cumprod` để tính giá,
cho biết giá cuối, giá cao nhất, mức sụt giảm lớn nhất từ đỉnh (max drawdown).

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Tự viết hàm báo cáo

Viết hàm `bao_cao(df, cot_nhom)` trả về DataFrame gồm: số hóa đơn, tổng doanh thu, doanh thu TB/hóa đơn,
tổng lợi nhuận gộp, biên lợi nhuận gộp (%) theo cột nhóm bất kỳ. Gọi hàm với `thanh_pho`, `thanh_toan`, `loai_khach`.

In [ ]:
# Viết code của bạn ở đây
